# 🔍 Azure 리소스 연결 테스트

이 노트북은 Azure AI Search 핸즈온을 위한 환경 설정을 검증합니다.

## 📋 테스트 항목

1. **Azure AI Search** - 검색 서비스 연결 확인
2. **Azure OpenAI Embedding 모델** - text-embedding-ada-002 모델 테스트
3. **Azure OpenAI Chat 모델** - gpt-4.1-mini 모델 테스트

## ⚙️ 사전 준비사항

- `.env` 파일이 프로젝트 루트에 생성되어 있어야 합니다
- 필수 환경 변수:
  - `SEARCH_ENDPOINT`: Azure AI Search 엔드포인트
  - `SEARCH_ADMIN_KEY`: Azure AI Search 관리 키
  - `AZURE_OPEN_AI_ENDPOINT`: Azure OpenAI Foundry 프로젝트 엔드포인트
  - `AZURE_OPEN_AI_KEY`: Azure OpenAI 프로젝트 키
  - `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`: Embedding 모델 배포 이름
  - `AZURE_OPENAI_CHAT_DEPLOYMENT`: Chat 모델 배포 이름

---

## 1️⃣ 라이브러리 임포트 및 환경 설정

필요한 Python 라이브러리를 임포트하고 경고 메시지를 필터링합니다.

In [13]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI
from dotenv import load_dotenv
import os


def make_aoai_client(api_version: str) -> AzureOpenAI:
    """
    Azure OpenAI 클라이언트를 생성합니다.

    - AZURE_OPEN_AI_KEY가 설정되어 있으면 키 인증 사용 (대부분의 워크숍 참가자)
    - 키가 없거나 빈 값이면 Entra ID(AAD) 토큰 인증으로 폴백
      (리소스에서 키 인증이 비활성화된 환경 대응; az login 필요)
    """
    endpoint = os.getenv("AZURE_OPEN_AI_ENDPOINT")
    key = os.getenv("AZURE_OPEN_AI_KEY")
    # AOAI_AUTH=aad 로 강제 AAD 사용 가능 (키가 .env에 남아 있지만 리소스에서 키 인증이 꺼진 경우)
    use_key = (
        os.getenv("AOAI_AUTH", "").lower() != "aad"
        and key and key.strip() and not key.startswith("your-")
    )
    if use_key:
        return AzureOpenAI(
            azure_endpoint=endpoint,
            api_key=key,
            api_version=api_version,
        )
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(),
        "https://cognitiveservices.azure.com/.default",
    )
    return AzureOpenAI(
        azure_endpoint=endpoint,
        azure_ad_token_provider=token_provider,
        api_version=api_version,
    )


print("✅ 라이브러리 임포트 완료")


✅ 라이브러리 임포트 완료


## 2️⃣ 환경 변수 로드

`.env` 파일에서 Azure 리소스 접속 정보를 로드합니다.

**💡 참고:** `.env` 파일을 수정한 경우, 이 셀을 다시 실행하면 변경사항이 반영됩니다.

In [14]:
# 환경 변수 로드 (override=True로 기존 값 덮어쓰기)
load_dotenv(override=True)

print("✅ 환경 변수 로드 완료")
print(f"📁 현재 작업 디렉토리: {os.getcwd()}")

✅ 환경 변수 로드 완료
📁 현재 작업 디렉토리: /Users/changjuahn/Repo/azure_aisearch_workshop/01-setup


## 3️⃣ 환경 변수 확인

필수 환경 변수가 올바르게 설정되어 있는지 확인합니다.

In [3]:
print("📋 환경 변수 확인 중...\n")

required_vars = {
    "SEARCH_ENDPOINT": os.getenv("SEARCH_ENDPOINT"),
    "SEARCH_ADMIN_KEY": os.getenv("SEARCH_ADMIN_KEY"),
    "AZURE_OPEN_AI_ENDPOINT": os.getenv("AZURE_OPEN_AI_ENDPOINT"),
    "AZURE_OPEN_AI_KEY": os.getenv("AZURE_OPEN_AI_KEY"),
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT": os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    "AZURE_OPENAI_CHAT_DEPLOYMENT": os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
}

all_set = True
for var_name, var_value in required_vars.items():
    if var_value:
        # 키 값은 일부만 표시
        if "KEY" in var_name:
            display_value = var_value[:8] + "..." if len(var_value) > 8 else var_value
        else:
            display_value = var_value
        print(f"✅ {var_name}: {display_value}")
    else:
        print(f"❌ {var_name}: 설정되지 않음")
        all_set = False

print()

if all_set:
    print("✅ 모든 필수 환경 변수가 설정되었습니다!")
else:
    print("⚠️  일부 환경 변수가 설정되지 않았습니다. .env 파일을 확인하세요.")

📋 환경 변수 확인 중...

✅ SEARCH_ENDPOINT: https://ai-search-foundry-iq-cj.search.windows.net
✅ SEARCH_ADMIN_KEY: 5ppmMD0D...
✅ AZURE_OPEN_AI_ENDPOINT: https://foundry-aisearch-models.openai.azure.com
✅ AZURE_OPEN_AI_KEY: F75btlP0...
✅ AZURE_OPENAI_EMBEDDING_DEPLOYMENT: text-embedding-3-large
✅ AZURE_OPENAI_CHAT_DEPLOYMENT: gpt-5.2

✅ 모든 필수 환경 변수가 설정되었습니다!


## 4️⃣ Azure AI Search 연결 테스트

Azure AI Search 서비스에 연결하여 인덱스 목록을 조회합니다.

In [4]:
search_ok = False
try:
    endpoint = os.getenv("SEARCH_ENDPOINT")
    key = os.getenv("SEARCH_ADMIN_KEY")
    
    if not endpoint or not key:
        print("❌ SEARCH_ENDPOINT 또는 SEARCH_ADMIN_KEY가 설정되지 않았습니다.")
    else:
        print("🔍 Azure AI Search 연결 테스트 중...")
        print(f"   Endpoint: {endpoint}")
        
        # SearchIndexClient로 연결 테스트
        credential = AzureKeyCredential(key)
        client = SearchIndexClient(endpoint=endpoint, credential=credential)
        
        # 인덱스 목록 조회로 연결 확인
        indexes = list(client.list_indexes())
        
        print(f"\n✅ Azure AI Search 연결 성공!")
        print(f"   기존 인덱스 수: {len(indexes)}")
        if indexes:
            print(f"   인덱스 목록: {[idx.name for idx in indexes]}")
        
        search_ok = True
        
except Exception as e:
    print(f"\n❌ Azure AI Search 연결 실패: {str(e)}")
    search_ok = False

🔍 Azure AI Search 연결 테스트 중...
   Endpoint: https://ai-search-foundry-iq-cj.search.windows.net

✅ Azure AI Search 연결 성공!
   기존 인덱스 수: 3
   인덱스 목록: ['image-similarity-poc', 'manufacturing-kb-index', 'products-index']


## 5️⃣ Azure OpenAI Embedding 모델 테스트

`text-embedding-3-large` 모델을 사용하여 텍스트 임베딩을 생성합니다.

In [15]:
embedding_ok = False
try:
    endpoint = os.getenv("AZURE_OPEN_AI_ENDPOINT")
    deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
    api_version = os.getenv("AZURE_OPENAI_EMBEDDING_API_VERSION", os.getenv("AZURE_OPENAI_API_VERSION"))
    
    if not endpoint:
        print("❌ AZURE_OPEN_AI_ENDPOINT가 설정되지 않았습니다.")
    else:
        print("🤖 Azure OpenAI Embedding 모델 테스트 중...")
        print(f"   Endpoint: {endpoint}")
        print(f"   Deployment: {deployment}")
        print(f"   API Version: {api_version}")

        client = make_aoai_client(api_version)
        
        # 임베딩 테스트
        response = client.embeddings.create(
            input="Azure AI Search 핸즈온 테스트",
            model=deployment
        )
        
        embedding_dim = len(response.data[0].embedding)
        
        print(f"\n✅ Embedding 모델 연결 성공!")
        print(f"   임베딩 차원: {embedding_dim}")
        print(f"   모델: {response.model}")
        
        embedding_ok = True
        
except Exception as e:
    print(f"\n❌ Embedding 모델 연결 실패: {str(e)}")
    embedding_ok = False


🤖 Azure OpenAI Embedding 모델 테스트 중...
   Endpoint: https://foundry-aisearch-models.openai.azure.com
   Deployment: text-embedding-3-large
   API Version: 2025-04-01-preview

✅ Embedding 모델 연결 성공!
   임베딩 차원: 3072
   모델: text-embedding-3-large


## 6️⃣ Azure OpenAI Chat 모델 테스트

Chat Completion 모델(`gpt-5.2`)의 연결을 테스트합니다. 간단한 대화형 요청을 보내 응답을 확인합니다.

In [16]:
chat_ok = False
try:
    endpoint = os.getenv("AZURE_OPEN_AI_ENDPOINT")
    deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
    api_version = os.getenv("AZURE_OPENAI_CHAT_API_VERSION")
    
    if not endpoint:
        print("❌ AZURE_OPEN_AI_ENDPOINT가 설정되지 않았습니다.")
    else:
        print("💬 Azure OpenAI Chat 모델 테스트 중...")
        print(f"   Endpoint: {endpoint}")
        print(f"   Deployment: {deployment}")

        client = make_aoai_client(api_version)
        
        # Chat Completion 테스트
        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
                {"role": "user", "content": "안녕하세요. Azure AI Search에 대해 간단히 설명해주세요."}
            ],
        )
        
        print(f"\n✅ Chat 모델 연결 성공!")
        print(f"   모델: {response.model}")
        print(f"   응답 내용:")
        print(f"   {response.choices[0].message.content[:200]}...")
        
        chat_ok = True
        
except Exception as e:
    print(f"\n❌ Chat 모델 연결 실패: {str(e)}")
    chat_ok = False


💬 Azure OpenAI Chat 모델 테스트 중...
   Endpoint: https://foundry-aisearch-models.openai.azure.com
   Deployment: gpt-5.2

✅ Chat 모델 연결 성공!
   모델: gpt-5.2-2025-12-11
   응답 내용:
   Azure AI Search(구 Azure Cognitive Search)는 **클라우드에서 제공되는 검색 서비스**로, 애플리케이션에 **고성능 전체 텍스트 검색**과 **AI 기반 정보 추출(인덱싱)** 기능을 쉽게 붙일 수 있게 해줍니다. 문서, 웹페이지, PDF, 이미지 메타데이터 같은 다양한 데이터에서 검색 가능한 인덱스를 만들어 **빠르고 정확하...


## 7️⃣ 테스트 결과 요약

모든 서비스의 연결 테스트 결과를 한눈에 확인할 수 있습니다.

In [7]:
print("\n" + "="*60)
print("📊 전체 테스트 결과 요약")
print("="*60)
print(f"Azure AI Search     : {'✅ 성공' if search_ok else '❌ 실패'}")
print(f"OpenAI Embedding    : {'✅ 성공' if embedding_ok else '❌ 실패'}")
print(f"OpenAI Chat         : {'✅ 성공' if chat_ok else '❌ 실패'}")
print("="*60)

if search_ok and embedding_ok and chat_ok:
    print("\n🎉 모든 서비스 연결이 정상입니다! 다음 단계로 진행하세요.")
else:
    print("\n⚠️  일부 서비스 연결에 문제가 있습니다. 환경 변수 및 네트워크 설정을 확인하세요.")


📊 전체 테스트 결과 요약
Azure AI Search     : ✅ 성공
OpenAI Embedding    : ❌ 실패
OpenAI Chat         : ❌ 실패

⚠️  일부 서비스 연결에 문제가 있습니다. 환경 변수 및 네트워크 설정을 확인하세요.


## 8️⃣ 패키지 버전 확인

**참고:** 본 워크숍의 모든 모듈(01~10)은 GA 버전(`azure-search-documents>=11.6.0`, 권장 `12.0.0`)으로 동작합니다.
Preview(`11.7.0b2` 등) 설치가 필요한 작업은 없습니다.

아래 셀을 실행하여 현재 설치된 패키지 버전을 확인합니다.

In [8]:
import subprocess
import sys

print("📦 현재 설치된 주요 패키지 버전 확인\n")

# 확인할 패키지 목록
packages_to_check = [
    "azure-search-documents",
    "openai",
    "azure-identity",
    "python-dotenv"
]

# 현재 설치된 버전 확인
installed_versions = {}
for package in packages_to_check:
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if line.startswith('Version:'):
                    version = line.split(':', 1)[1].strip()
                    installed_versions[package] = version
                    
                    # Preview 버전 체크
                    if 'b' in version or 'a' in version or 'rc' in version:
                        print(f"⚠️  {package}: {version} (Preview 버전)")
                    else:
                        print(f"✅ {package}: {version}")
                    break
        else:
            print(f"❌ {package}: 설치되지 않음")
            installed_versions[package] = None
    except Exception as e:
        print(f"❌ {package} 확인 실패: {str(e)}")
        installed_versions[package] = None

# azure-search-documents 버전 체크 및 복구 가이드
print("\n" + "="*60)
asd_version = installed_versions.get("azure-search-documents")

if asd_version and ('b' in asd_version or 'a' in asd_version):
    print("⚠️  Preview 패키지가 설치되어 있습니다!")
    print("="*60)
    print("\n📌 현재 상태:")
    print(f"   - azure-search-documents: {asd_version} (Preview)")
    print("\n🔄 모든 모듈(01~10)은 GA 버전으로 동작합니다. GA 버전으로 복구하시겠습니까?")
    print("\n다음 명령을 실행하세요:")
    print("   !pip install --upgrade -r ../requirements.txt")
    print("\n또는 개별 설치:")
    print("   !pip install --upgrade 'azure-search-documents>=11.6.0'")
    print("\n⚠️ 설치 후 Jupyter 커널을 재시작하세요:")
    print("   Ctrl + Shift + P → 'Jupyter: Restart Kernel'")
    
    # 자동 복구 옵션
    print("\n" + "="*60)
    user_input = input("지금 바로 복구하시겠습니까? (y/N): ").strip().lower()
    
    if user_input == 'y':
        print("\n🔄 GA 버전으로 복구 중...")
        try:
            result = subprocess.run(
                [sys.executable, "-m", "pip", "install", "--upgrade", "azure-search-documents>=11.6.0"],
                capture_output=True,
                text=True
            )
            if result.returncode == 0:
                print("✅ 패키지 복구 완료!")
                print("⚠️ Jupyter 커널을 재시작하세요: Ctrl + Shift + P → 'Jupyter: Restart Kernel'")
            else:
                print(f"❌ 복구 실패: {result.stderr}")
        except Exception as e:
            print(f"❌ 복구 중 오류 발생: {str(e)}")
    else:
        print("\n복구를 건너뛰었습니다. 필요시 위의 명령을 직접 실행하세요.")
        
elif asd_version:
    print("✅ GA 버전이 설치되어 있습니다.")
    print("="*60)
    print(f"   - azure-search-documents: {asd_version}")
    print("\n모든 모듈(01~10) 실습을 진행할 수 있습니다.")
else:
    print("❌ azure-search-documents가 설치되지 않았습니다.")
    print("="*60)
    print("\n다음 명령으로 설치하세요:")
    print("   !pip install -r ../requirements.txt")

print("\n" + "="*60)

📦 현재 설치된 주요 패키지 버전 확인

✅ azure-search-documents: 11.6.0
✅ openai: 2.16.0
✅ azure-identity: 1.25.1
✅ python-dotenv: 1.2.1

✅ GA 버전이 설치되어 있습니다.
   - azure-search-documents: 11.6.0

01-08 모듈 실습을 진행할 수 있습니다.

💡 09-foundryiq 모듈 실습 시에는 Preview 버전 설치가 필요합니다:
   !pip install --upgrade azure-search-documents==11.7.0b2



---

## 🎯 다음 단계

연결 테스트가 성공적으로 완료되었다면, 다음 모듈로 이동하세요:

1. **02-indexing**: Azure AI Search 인덱스 생성 및 데이터 업로드
2. **03-search**: 다양한 검색 방법 실습 (Keyword, Vector, Hybrid, Semantic)
3. **04-scoring**: Score Profile을 활용한 검색 결과 최적화
4. **05-ai-skills**: AI Skills를 활용한 데이터 보강

---

### 💡 문제 해결 팁

**연결 실패 시 확인 사항:**
- `.env` 파일이 올바른 위치에 있는지 확인
- 환경 변수가 정확하게 설정되었는지 확인
- Azure Portal에서 리소스가 정상적으로 생성되었는지 확인
- 방화벽 및 네트워크 설정 확인
- API Key가 유효한지 확인 (만료되지 않았는지)

---

## 🧭 다음 단계

| ⬅️ 이전 | 🏠 목차 | ➡️ 다음 |
|:---------|:-------:|--------:|
| - | [워크숍 홈](../README.md) | [Lab 02-1: 인덱스 생성](../02-keyword_search/01-create_index.ipynb) |